## 1. Load Property Events

Property Events are owned by another NorthStar team and stored in Azure Blob Storage.

A OneLake shortcut gives us access to the files without creating another copy of the data.


In [ ]:
import pandas as pd
from pathlib import Path

files = sorted(
    Path("/lakehouse/default/Files/property-events").glob("*.json")
)

if not files:
    raise FileNotFoundError(
        "No event files found. Attach the default Lakehouse, restart the session, "
        "and upload the extracted JSON files into Files/property-events."
    )

events_raw = pd.concat(
    [pd.read_json(file, lines=True) for file in files],
    ignore_index=True
)

print(f"Rows loaded: {len(events_raw)}")
events_raw.head()


## 2. Standardize Property Events

Operational systems don't always represent the same business event consistently.

Standardize event types and timestamps so downstream consumers receive predictable data.


In [ ]:
events = events_raw.copy()

event_type_map = {
    "ROOM_OOS": "ROOM_OUT_OF_SERVICE",
    "room-oos": "ROOM_OUT_OF_SERVICE",
    "Room Out Of Service": "ROOM_OUT_OF_SERVICE",
    "room oos": "ROOM_OUT_OF_SERVICE",
    "MAINTENANCE": "MAINTENANCE",
    "maintenance": "MAINTENANCE",
    "AMENITY_CLOSED": "AMENITY_CLOSED",
    "Amenity Closed": "AMENITY_CLOSED",
    "ROOM_RETURNED_TO_SERVICE": "ROOM_RETURNED_TO_SERVICE",
    "room-returned-to-service": "ROOM_RETURNED_TO_SERVICE"
}

events["event_type_standardized"] = (
    events["event_type"]
    .map(event_type_map)
    .fillna(events["event_type"].str.upper())
)

events["event_timestamp"] = pd.to_datetime(
    events["event_time"],
    format="mixed",
    errors="coerce",
    utc=True
)

events["event_date"] = events["event_timestamp"].dt.date

events[
    [
        "event_id",
        "event_type",
        "event_type_standardized",
        "event_timestamp"
    ]
].head(10)


## 3. Validate and Enrich

The Property table loaded from Azure SQL is our trusted source for NorthStar properties.

Validate each event against that data and enrich valid events with the property name.


In [ ]:
import duckdb

properties = duckdb.sql("""
    SELECT property_code, property_name
    FROM delta_scan('/lakehouse/default/Tables/landing/Property')
""").df()

events_validated = events.merge(
    properties,
    on="property_code",
    how="left"
)

events_validated["is_valid_property"] = (
    events_validated["property_name"].notna()
)

valid_events = events_validated[
    events_validated["is_valid_property"]
].copy()

quarantine_events = events_validated[
    ~events_validated["is_valid_property"]
].copy()

print(f"Total events:       {len(events_validated)}")
print(f"Valid events:       {len(valid_events)}")
print(f"Quarantined events: {len(quarantine_events)}")


## 4. Prepare Reusable Data

Valid records are shaped into a consistent dataset for downstream consumers.

Records that fail validation are preserved separately so they can be investigated and corrected.


In [ ]:
valid_output = valid_events[
    [
        "event_id",
        "property_code",
        "property_name",
        "event_timestamp",
        "event_date",
        "event_type_standardized",
        "room_number",
        "severity",
        "description",
        "source_system"
    ]
].copy()

valid_output = valid_output.rename(
    columns={"event_type_standardized": "event_type"}
).reset_index(drop=True)

quarantine_output = quarantine_events[
    [
        "event_id",
        "property_code",
        "event_time",
        "event_type",
        "room_number",
        "severity",
        "description",
        "source_system"
    ]
].copy().reset_index(drop=True)

quarantine_output["event_time"] = (
    quarantine_output["event_time"].astype(str)
)

quarantine_output["quarantine_reason"] = (
    "Invalid property_code"
)

valid_output.head()


## 5. Publish Reusable Data

Publish validated events as a reusable Delta table and preserve failed records in a quarantine table.

These tables become the trusted output of the Data Engineering process.


In [ ]:
from deltalake import write_deltalake
import notebookutils

context = notebookutils.runtime.context
lakehouse_id = context.get("defaultLakehouseId")
workspace_id = context.get("defaultLakehouseWorkspaceId")

if not lakehouse_id or not workspace_id:
    raise RuntimeError("Attach a schema-enabled Lakehouse as the default and restart the session.")

onelake_endpoint = (
    notebookutils.conf.get("trident.onelake.endpoint")
    .removeprefix("https://")
    .rstrip("/")
)
base_path = f"abfss://{workspace_id}@{onelake_endpoint}/{lakehouse_id}"

write_deltalake(
    f"{base_path}/Tables/processed/PropertyEvent",
    valid_output,
    mode="overwrite"
)

write_deltalake(
    f"{base_path}/Tables/quarantine/PropertyEvent",
    quarantine_output,
    mode="overwrite"
)

print("Property Event processing complete.")
print(f"Processed:   {len(valid_output)}")
print(f"Quarantined: {len(quarantine_output)}")
